Local fetch

In [ ]:
import pandas as pd
import numpy as np


# Load Data
columns = ['review', 'sentiment']
augment_url = "diversity_sampling/dataset/coresets/augment.csv"
augment = pd.read_csv(augment_url)[columns]

# label normalization
augment['sentiment'] = (augment['sentiment'] == 'positive').astype(int)

DB fetch alternative 

In [ ]:
import pandas as pd
import numpy as np
from diversity_sampling.database.api import get_augment_set

test_set = pd.DataFrame(await get_augment_set())
test_set

In [ ]:
# Test set construction
test_size = 0.1
test_len = int(len(augment) * (test_size))
test_set = augment.tail(test_len)
test_set.to_csv("diversity_sampling/dataset/test.csv", index=False)
train_len = int(len(augment) * (1-test_size)) +1

# Save the rest for training
real_train_full = augment.head(train_len)

# Experimental Setup
random_url = "diversity_sampling/dataset/random_sampling/"
diverse_url = "diversity_sampling/dataset/diversity_sampling/"


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

def calculate_k(r: float, train_set_len: int):
    """
    With r = synthetic-real ratio
    Calculate needed samples, denotes as k,  to add to train set such that k / (train_set_len + k) = r 
    
    """
    factor = r / (1 -r) if (1 - r) > 0 else 1.0
    k = int(factor * (train_set_len))
    return k
    
    
def random_selection(k: int, synthetic_data: pd.DataFrame):
    synthetic_add = synthetic_data.sample(k)
    
    return synthetic_add


def diversity_sampling(n_target, synthetic_data, text_column='review', n_clusters=None, embedding_model=None):
    model = embedding_model if embedding_model else SentenceTransformer('all-MiniLM-L6-v2')
    
    sentences = synthetic_data[text_column].tolist()
    embeddings = model.encode(sentences, show_progress_bar=True)
    
    if n_clusters is None:
        n_clusters = max(1, n_target // 5) 
    
    kmeans = KMeans(n_clusters=n_clusters, init='k-means++', random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings)
    centroids = kmeans.cluster_centers_
    
    samples_per_cluster = n_target // n_clusters
    
    selected_indices = []
    
    for i in range(n_clusters):
        cluster_indices = np.where(cluster_labels == i)[0]
        
        if len(cluster_indices) == 0:
            continue
            
        cluster_embeddings = embeddings[cluster_indices]
        
        centroid = centroids[i].reshape(1, -1)
        similarities = cosine_similarity(cluster_embeddings, centroid).flatten()
        
        top_local_indices = similarities.argsort()[-samples_per_cluster:]
        
        selected_indices.extend(cluster_indices[top_local_indices])
    
    if len(selected_indices) < n_target:
        diff = n_target - len(selected_indices)
        remaining_pool = list(set(range(len(synthetic_data))) - set(selected_indices))
        selected_indices.extend(remaining_pool[:diff])

    return synthetic_data.iloc[selected_indices]
    

Local fetch

In [ ]:
# Synthetic set
synthetic_url = "diversity_sampling/dataset/augment_sets/high_quality.csv"
synthetic = pd.read_csv(synthetic_url)

# Predefined-ratios
ratios = [x / 10 for x in range(11)]

DB fetch

In [ ]:
import pandas as pd
import numpy as np
from diversity_sampling.database.api import get_high_quality_synthetic_set

synthetic_url = "diversity_sampling/dataset/augment_sets/high_quality.csv"
synthetic = pd.DataFrame(await get_high_quality_synthetic_set)

# Predefined-ratios
ratios = [x / 10 for x in range(11)]

# Random Sampling

In [ ]:
# Random Sampling
random_url = "diversity_sampling/dataset/random_sampling/"

for r in ratios:
    set_name = f"random_r_{str(r)}0.csv"
    k = calculate_k(r, len(real_train_full))
    
    if k <= len(synthetic):
        synthetic_adds_up = random_selection(k, synthetic)
    
        training_set = pd.concat([real_train_full, synthetic_adds_up], axis=0).reset_index(drop=True)
        training_set.to_csv(f"{random_url}/{set_name}")
        
        print(f"Ratio {r} generation succeed.\nTrain set size: {len(training_set)}, Synthetic size: {len(synthetic_adds_up)}\n")
    else:
        print(f"Ratio {r} generation interrupt.\nData insufficient, needed {k}, got {len(synthetic)}.\n")

# Diversity-Oriented Sampling

In [ ]:
diversity_url = "diversity_sampling/dataset/diversity_sampling/"

n_clusters = 10
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

for r in ratios:
    set_name = f"diverse_r_{str(r)}0.csv"
    k = calculate_k(r, len(real_train_full))

    if k <= len(synthetic):
        synthetic_adds_up = diversity_sampling(
            n_target=k, 
            synthetic_data=synthetic,
            n_clusters=n_clusters,
            embedding_model=embedding_model
        )
        
        training_set = pd.concat([real_train_full, synthetic_adds_up], axis=0).reset_index(drop=True)
        training_set.to_csv(f"{diversity_url}/{set_name}")
        
        print(f"Ratio {r} generation succeed.\nTrain set size: {len(training_set)}, Synthetic size: {len(synthetic_adds_up)}\n")
    else:
        print(f"Ratio {r} generation interrupt.\nData insufficient, needed {k}, got {len(synthetic)}.\n")
    